# 시험 직전 초보자 복습팩 실습 노트북 — Filled Version

이 파일은 원본 Colab 실습을 시험 직전용으로 바로 실행할 수 있게 만든 **정답 채움판**입니다.

사용법:

1. Colab에 업로드한다.
2. 맨 위 준비 셀을 먼저 실행한다.
3. SQL 20문제와 Python/pandas/통계 40문제를 위에서부터 실행한다.
4. 안 외워지는 문제는 코드 전체가 아니라 `패턴 이름`만 외운다.

핵심 암기:

```text
SELECT = 보여줄 컬럼
FROM = 기준 테이블
JOIN = 붙일 테이블
ON = 연결 조건
WHERE = 행 조건
GROUP BY = 묶는 기준
HAVING = 묶은 뒤 조건
ORDER BY = 정렬
LIMIT = 개수 제한
```


## 0. 준비 셀

반드시 먼저 실행하세요.


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2_contingency

# ------------------------------------------------------------
# 0. SQLite 준비
# ------------------------------------------------------------
conn = sqlite3.connect(":memory:")

products = pd.DataFrame([
    {"productCode":"S10_1678","productName":"1969 Harley Davidson","productLine":"Motorcycles","MSRP":95.70},
    {"productCode":"S10_1949","productName":"1952 Alpine Renault","productLine":"Classic Cars","MSRP":214.30},
    {"productCode":"S10_2016","productName":"1996 Moto Guzzi","productLine":"Motorcycles","MSRP":118.94},
    {"productCode":"S12_1099","productName":"1968 Ford Mustang","productLine":"Classic Cars","MSRP":194.57},
    {"productCode":"S18_2795","productName":"1928 Mercedes-Benz","productLine":"Vintage Cars","MSRP":168.75},
    {"productCode":"S24_2000","productName":"Unordered Sample","productLine":"Trucks and Buses","MSRP":80.00},
])

customers = pd.DataFrame([
    {"customerNumber":101,"customerName":"Atelier graphique","country":"France","creditLimit":21000},
    {"customerNumber":102,"customerName":"Signal Gift Stores","country":"USA","creditLimit":71800},
    {"customerNumber":103,"customerName":"Australian Collectors","country":"Australia","creditLimit":117300},
    {"customerNumber":104,"customerName":"La Rochelle Gifts","country":"France","creditLimit":118200},
    {"customerNumber":105,"customerName":"Tokyo Collectables","country":"Japan","creditLimit":94400},
    {"customerNumber":106,"customerName":"No Order Customer","country":"USA","creditLimit":50000},
])

orders = pd.DataFrame([
    {"orderNumber":5001,"orderDate":"2003-01-15","customerNumber":101,"status":"Shipped"},
    {"orderNumber":5002,"orderDate":"2003-02-20","customerNumber":102,"status":"Shipped"},
    {"orderNumber":5003,"orderDate":"2004-03-10","customerNumber":102,"status":"Resolved"},
    {"orderNumber":5004,"orderDate":"2004-04-18","customerNumber":103,"status":"Shipped"},
    {"orderNumber":5005,"orderDate":"2004-05-22","customerNumber":104,"status":"Cancelled"},
    {"orderNumber":5006,"orderDate":"2004-06-11","customerNumber":105,"status":"Shipped"},
    {"orderNumber":5007,"orderDate":"2005-01-09","customerNumber":104,"status":"Shipped"},
])

orderdetails = pd.DataFrame([
    {"orderNumber":5001,"productCode":"S10_1949","quantityOrdered":2,"priceEach":200.0},
    {"orderNumber":5001,"productCode":"S12_1099","quantityOrdered":1,"priceEach":180.0},
    {"orderNumber":5002,"productCode":"S10_1678","quantityOrdered":5,"priceEach":90.0},
    {"orderNumber":5003,"productCode":"S10_2016","quantityOrdered":3,"priceEach":110.0},
    {"orderNumber":5003,"productCode":"S18_2795","quantityOrdered":2,"priceEach":160.0},
    {"orderNumber":5004,"productCode":"S10_1949","quantityOrdered":1,"priceEach":205.0},
    {"orderNumber":5004,"productCode":"S24_2000","quantityOrdered":7,"priceEach":80.0},
    {"orderNumber":5005,"productCode":"S12_1099","quantityOrdered":2,"priceEach":190.0},
    {"orderNumber":5006,"productCode":"S10_1678","quantityOrdered":4,"priceEach":95.0},
    {"orderNumber":5007,"productCode":"S18_2795","quantityOrdered":3,"priceEach":170.0},
])

payments = pd.DataFrame([
    {"customerNumber":101,"paymentDate":"2003-01-20","amount":580.0},
    {"customerNumber":102,"paymentDate":"2003-02-28","amount":450.0},
    {"customerNumber":102,"paymentDate":"2004-03-18","amount":650.0},
    {"customerNumber":103,"paymentDate":"2004-04-25","amount":765.0},
    {"customerNumber":104,"paymentDate":"2005-01-15","amount":510.0},
    {"customerNumber":105,"paymentDate":"2004-06-20","amount":380.0},
])

for name, df in {
    "products": products,
    "customers": customers,
    "orders": orders,
    "orderdetails": orderdetails,
    "payments": payments,
}.items():
    df.to_sql(name, conn, index=False, if_exists="replace")

def run_sql(query):
    """SQL 문자열을 실행하고 pandas DataFrame으로 보여준다."""
    return pd.read_sql_query(query, conn)

# ------------------------------------------------------------
# 1. pandas / 통계용 데이터
# ------------------------------------------------------------
exam_df = pd.DataFrame([
    {"country":"Korea","channel":"online","monthly_sales":120.0,"hire_type":"new","final_pass":"pass","visits":1200,"orders":35,"order_date":"2026-01-10","ad_cost":100},
    {"country":"Korea","channel":"offline","monthly_sales":98.0,"hire_type":"new","final_pass":"fail","visits":900,"orders":20,"order_date":"2026-01-16","ad_cost":80},
    {"country":"Japan","channel":"online","monthly_sales":135.0,"hire_type":"career","final_pass":"pass","visits":1500,"orders":44,"order_date":"2026-02-11","ad_cost":130},
    {"country":"USA","channel":"offline","monthly_sales":88.0,"hire_type":"career","final_pass":"pass","visits":760,"orders":18,"order_date":"2026-02-13","ad_cost":70},
    {"country":"USA","channel":"online","monthly_sales":142.0,"hire_type":"new","final_pass":"pass","visits":1800,"orders":52,"order_date":"2026-03-02","ad_cost":150},
    {"country":"France","channel":"offline","monthly_sales":105.0,"hire_type":"career","final_pass":"fail","visits":840,"orders":22,"order_date":"2026-03-18","ad_cost":90},
    {"country":"France","channel":"online","monthly_sales":128.0,"hire_type":"new","final_pass":"pass","visits":1320,"orders":39,"order_date":"2026-04-05","ad_cost":120},
    {"country":"Japan","channel":"offline","monthly_sales":92.0,"hire_type":"career","final_pass":"fail","visits":810,"orders":19,"order_date":"2026-04-20","ad_cost":75},
])

display(run_sql("SELECT name FROM sqlite_master WHERE type='table';"))
display(exam_df.head())


## 1. SQL 20문제 정답 채움판

읽는 순서:

```text
FROM/JOIN → ON → SELECT → WHERE → GROUP BY → SUM/AVG/COUNT → ORDER BY
```


### S01. customers 전체 조회


In [ ]:
run_sql("""
SELECT *
FROM customers;
""")


### S02. 국가별 고객 수


In [ ]:
run_sql("""
SELECT
    country,
    COUNT(*) AS customer_count
FROM customers
GROUP BY country
ORDER BY customer_count DESC;
""")


### S03. 2004년 주문일별 주문 수


In [ ]:
run_sql("""
SELECT
    orderDate,
    COUNT(*) AS order_count_2004
FROM orders
WHERE strftime('%Y', orderDate) = '2004'
GROUP BY orderDate
ORDER BY orderDate;
""")


### S04. 제품 라인별 평균 MSRP


In [ ]:
run_sql("""
SELECT
    productLine,
    ROUND(AVG(MSRP), 2) AS avg_msrp
FROM products
GROUP BY productLine
ORDER BY avg_msrp DESC;
""")


### S05. 국가별 총 주문 금액


In [ ]:
run_sql("""
SELECT
    c.country,
    ROUND(SUM(od.quantityOrdered * od.priceEach), 2) AS total_sales
FROM customers AS c
JOIN orders AS o
    ON c.customerNumber = o.customerNumber
JOIN orderdetails AS od
    ON o.orderNumber = od.orderNumber
GROUP BY c.country
ORDER BY total_sales DESC;
""")


### S06. 고객별 주문 수


In [ ]:
run_sql("""
SELECT
    c.customerName,
    COUNT(o.orderNumber) AS order_count
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customerNumber = o.customerNumber
GROUP BY c.customerNumber, c.customerName
ORDER BY order_count DESC;
""")


### S07. 주문 수가 2건 이상인 고객


In [ ]:
run_sql("""
SELECT
    c.customerName,
    COUNT(o.orderNumber) AS order_count
FROM customers AS c
JOIN orders AS o
    ON c.customerNumber = o.customerNumber
GROUP BY c.customerNumber, c.customerName
HAVING COUNT(o.orderNumber) >= 2
ORDER BY order_count DESC;
""")


### S08. 주문한 적 있는 고객만 조회 - IN 서브쿼리


In [ ]:
run_sql("""
SELECT
    customerNumber,
    customerName,
    country
FROM customers
WHERE customerNumber IN (
    SELECT customerNumber
    FROM orders
)
ORDER BY customerNumber;
""")


### S09. 주문이 없는 고객 찾기 - LEFT JOIN


In [ ]:
run_sql("""
SELECT
    c.customerNumber,
    c.customerName,
    c.country
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customerNumber = o.customerNumber
WHERE o.orderNumber IS NULL;
""")


### S10. 고객 신용한도 순위 - RANK


In [ ]:
run_sql("""
SELECT
    customerName,
    country,
    creditLimit,
    RANK() OVER (ORDER BY creditLimit DESC) AS credit_rank
FROM customers
ORDER BY credit_rank;
""")


### S11. 상품라인별 주문수량 합계


In [ ]:
run_sql("""
SELECT
    p.productLine,
    SUM(od.quantityOrdered) AS total_quantity
FROM products AS p
JOIN orderdetails AS od
    ON p.productCode = od.productCode
GROUP BY p.productLine
ORDER BY total_quantity DESC;
""")


### S12. 주문별 총액


In [ ]:
run_sql("""
SELECT
    orderNumber,
    ROUND(SUM(quantityOrdered * priceEach), 2) AS order_total
FROM orderdetails
GROUP BY orderNumber
ORDER BY order_total DESC;
""")


### S13. 평균 주문총액보다 큰 주문


In [ ]:
run_sql("""
WITH order_totals AS (
    SELECT
        orderNumber,
        SUM(quantityOrdered * priceEach) AS order_total
    FROM orderdetails
    GROUP BY orderNumber
)
SELECT
    orderNumber,
    ROUND(order_total, 2) AS order_total
FROM order_totals
WHERE order_total > (
    SELECT AVG(order_total)
    FROM order_totals
)
ORDER BY order_total DESC;
""")


### S14. 제품별 판매액


In [ ]:
run_sql("""
SELECT
    p.productName,
    p.productLine,
    ROUND(SUM(od.quantityOrdered * od.priceEach), 2) AS product_sales
FROM products AS p
JOIN orderdetails AS od
    ON p.productCode = od.productCode
GROUP BY p.productCode, p.productName, p.productLine
ORDER BY product_sales DESC;
""")


### S15. 주문 상태별 주문 수


In [ ]:
run_sql("""
SELECT
    status,
    COUNT(*) AS order_count
FROM orders
GROUP BY status
ORDER BY order_count DESC;
""")


### S16. CASE로 신용한도 등급


In [ ]:
run_sql("""
SELECT
    customerName,
    creditLimit,
    CASE
        WHEN creditLimit >= 100000 THEN 'high'
        WHEN creditLimit >= 70000 THEN 'mid'
        ELSE 'low'
    END AS credit_group
FROM customers
ORDER BY creditLimit DESC;
""")


### S17. NTILE로 4구간 나누기


In [ ]:
run_sql("""
SELECT
    customerName,
    creditLimit,
    NTILE(4) OVER (ORDER BY creditLimit DESC) AS credit_quartile
FROM customers
ORDER BY credit_quartile, creditLimit DESC;
""")


### S18. 제품별 평균 주문 단가


In [ ]:
run_sql("""
SELECT
    productCode,
    ROUND(AVG(priceEach), 2) AS avg_price
FROM orderdetails
GROUP BY productCode
ORDER BY avg_price DESC;
""")


### S19. CTE로 국가별 매출 TOP 3


In [ ]:
run_sql("""
WITH country_sales AS (
    SELECT
        c.country,
        SUM(od.quantityOrdered * od.priceEach) AS total_sales
    FROM customers AS c
    JOIN orders AS o
        ON c.customerNumber = o.customerNumber
    JOIN orderdetails AS od
        ON o.orderNumber = od.orderNumber
    GROUP BY c.country
)
SELECT
    country,
    ROUND(total_sales, 2) AS total_sales
FROM country_sales
ORDER BY total_sales DESC
LIMIT 3;
""")


### S20. UNION ALL로 연도별 주문 고객 합치기


In [ ]:
run_sql("""
SELECT
    '2003' AS year_label,
    customerNumber
FROM orders
WHERE strftime('%Y', orderDate) = '2003'

UNION ALL

SELECT
    '2004' AS year_label,
    customerNumber
FROM orders
WHERE strftime('%Y', orderDate) = '2004';
""")


## 2. Python / pandas / 그래프 / 통계 40문제 정답 채움판

시험 직전에는 전부 외우려 하지 말고 아래 패턴만 잡으세요.

```text
df.head()
df.info()
df.isna().sum()
df.groupby()
pd.crosstab()
pd.pivot_table()
pd.to_datetime()
plt.scatter()
plt.hist()
stats.ttest_1samp()
chi2_contingency()
```


### P01. 데이터프레임 복사와 앞부분 확인


In [ ]:
df = exam_df.copy()
display(df.head())


### P02. 컬럼 정보 확인


In [ ]:
df = exam_df.copy()
df.info()


### P03. 결측치 개수 확인


In [ ]:
df = exam_df.copy()
display(df.isna().sum())


### P04. 기초 통계량 확인


In [ ]:
df = exam_df.copy()
display(df.describe())


### P05. country별 빈도


In [ ]:
df = exam_df.copy()
display(df['country'].value_counts())


### P06. channel별 평균 monthly_sales


In [ ]:
df = exam_df.copy()
display(df.groupby('channel')['monthly_sales'].mean())


### P07. country별 평균 visits/orders/ad_cost


In [ ]:
df = exam_df.copy()
display(df.groupby('country')[['visits', 'orders', 'ad_cost']].mean())


### P08. order_date를 datetime으로 변환


In [ ]:
df = exam_df.copy()
df['order_date'] = pd.to_datetime(df['order_date'])
display(df.dtypes)


### P09. 월 컬럼 만들기


In [ ]:
df = exam_df.copy()
df['order_date'] = pd.to_datetime(df['order_date'])
df['month'] = df['order_date'].dt.month
display(df[['order_date', 'month']])


### P10. 전환율 conversion_rate 만들기


In [ ]:
df = exam_df.copy()
df['conversion_rate'] = df['orders'] / df['visits']
display(df[['country', 'orders', 'visits', 'conversion_rate']])


### P11. 광고비 대비 주문수 orders_per_ad 만들기


In [ ]:
df = exam_df.copy()
df['orders_per_ad'] = df['orders'] / df['ad_cost']
display(df[['country', 'orders', 'ad_cost', 'orders_per_ad']])


### P12. monthly_sales가 120 이상인 행


In [ ]:
df = exam_df.copy()
display(df[df['monthly_sales'] >= 120])


### P13. online 채널만 필터링


In [ ]:
df = exam_df.copy()
display(df[df['channel'] == 'online'])


### P14. pass인 사람만 보기


In [ ]:
df = exam_df.copy()
display(df[df['final_pass'] == 'pass'])


### P15. country와 channel별 평균 monthly_sales


In [ ]:
df = exam_df.copy()
display(df.groupby(['country', 'channel'])['monthly_sales'].mean())


### P16. pivot_table로 country별 channel 평균


In [ ]:
df = exam_df.copy()
display(pd.pivot_table(df, index='country', columns='channel', values='monthly_sales', aggfunc='mean'))


### P17. crosstab으로 hire_type과 final_pass 빈도


In [ ]:
df = exam_df.copy()
display(pd.crosstab(df['hire_type'], df['final_pass']))


### P18. 상관계수 확인


In [ ]:
df = exam_df.copy()
display(df[['monthly_sales', 'visits', 'orders', 'ad_cost']].corr())


### P19. 산점도 그리기


In [ ]:
df = exam_df.copy()
plt.figure()
plt.scatter(df['visits'], df['orders'])
plt.xlabel('visits')
plt.ylabel('orders')
plt.title('visits vs orders')
plt.show()


### P20. 막대그래프 그리기


In [ ]:
df = exam_df.copy()
country_sales = df.groupby('country')['monthly_sales'].mean()
plt.figure()
country_sales.plot(kind='bar')
plt.xlabel('country')
plt.ylabel('avg monthly_sales')
plt.title('Average monthly_sales by country')
plt.show()


### P21. 히스토그램 그리기


In [ ]:
df = exam_df.copy()
plt.figure()
plt.hist(df['monthly_sales'], bins=5)
plt.xlabel('monthly_sales')
plt.ylabel('count')
plt.title('monthly_sales distribution')
plt.show()


### P22. 평균, 중앙값, 표준편차


In [ ]:
df = exam_df.copy()
print('mean:', df['monthly_sales'].mean())
print('median:', df['monthly_sales'].median())
print('std:', df['monthly_sales'].std())


### P23. 사분위수와 IQR


In [ ]:
df = exam_df.copy()
q1 = df['monthly_sales'].quantile(0.25)
q3 = df['monthly_sales'].quantile(0.75)
iqr = q3 - q1
print('Q1:', q1)
print('Q3:', q3)
print('IQR:', iqr)


### P24. z검정 직접 계산


In [ ]:
xbar = 153
mu0 = 150
sigma = 12
n = 100
se = sigma / np.sqrt(n)
z = (xbar - mu0) / se
print('SE:', se)
print('z:', z)
print('판단:', 'H0 기각' if z > 1.645 else 'H0 기각 못함')


### P25. 단일표본 t-test


In [ ]:
df = exam_df.copy()
t_stat, p_value = stats.ttest_1samp(df['monthly_sales'], popmean=100)
print('t:', t_stat)
print('p-value:', p_value)


### P26. online/offline 평균 비교 t-test


In [ ]:
df = exam_df.copy()
online = df[df['channel'] == 'online']['monthly_sales']
offline = df[df['channel'] == 'offline']['monthly_sales']
t_stat, p_value = stats.ttest_ind(online, offline, equal_var=False)
print('t:', t_stat)
print('p-value:', p_value)


### P27. 카이제곱 검정


In [ ]:
df = exam_df.copy()
table = pd.crosstab(df['hire_type'], df['final_pass'])
chi2, p, dof, expected = chi2_contingency(table)
display(table)
print('chi2:', chi2)
print('p-value:', p)


### P28. 조건부 확률 P(pass | new)


In [ ]:
df = exam_df.copy()
new_df = df[df['hire_type'] == 'new']
prob = (new_df['final_pass'] == 'pass').mean()
print('P(pass | new) =', prob)


### P29. 조건부 확률 P(online | pass)


In [ ]:
df = exam_df.copy()
pass_df = df[df['final_pass'] == 'pass']
prob = (pass_df['channel'] == 'online').mean()
print('P(online | pass) =', prob)


### P30. apply로 sales_level 만들기


In [ ]:
df = exam_df.copy()
def label_sales(x):
    if x >= 130:
        return 'high'
    elif x >= 100:
        return 'mid'
    else:
        return 'low'

df['sales_level'] = df['monthly_sales'].apply(label_sales)
display(df[['monthly_sales', 'sales_level']])


### P31. np.where로 pass_flag 만들기


In [ ]:
df = exam_df.copy()
df['pass_flag'] = np.where(df['final_pass'] == 'pass', 1, 0)
display(df[['final_pass', 'pass_flag']])


### P32. 정렬하기


In [ ]:
df = exam_df.copy()
display(df.sort_values('monthly_sales', ascending=False))


### P33. 상위 3개 행


In [ ]:
df = exam_df.copy()
display(df.sort_values('monthly_sales', ascending=False).head(3))


### P34. country별 rank 만들기


In [ ]:
df = exam_df.copy()
df['sales_rank_in_country'] = df.groupby('country')['monthly_sales'].rank(ascending=False)
display(df[['country', 'monthly_sales', 'sales_rank_in_country']])


### P35. groupby + agg


In [ ]:
df = exam_df.copy()
summary = df.groupby('channel').agg(
    avg_sales=('monthly_sales', 'mean'),
    total_orders=('orders', 'sum'),
    avg_visits=('visits', 'mean')
)
display(summary)


### P36. query 사용


In [ ]:
df = exam_df.copy()
display(df.query("channel == 'online' and monthly_sales >= 120"))


### P37. isin 사용


In [ ]:
df = exam_df.copy()
display(df[df['country'].isin(['Korea', 'USA'])])


### P38. 문자열 포함 검색


In [ ]:
df = exam_df.copy()
display(df[df['country'].str.contains('a', case=False)])


### P39. csv 저장 예시


In [ ]:
df = exam_df.copy()
df.to_csv('exam_df_output.csv', index=False)
print('saved: exam_df_output.csv')


### P40. 최종 요약 테이블 만들기


In [ ]:
df = exam_df.copy()
df['conversion_rate'] = df['orders'] / df['visits']
final_summary = df.groupby('country').agg(
    avg_sales=('monthly_sales', 'mean'),
    total_orders=('orders', 'sum'),
    avg_conversion=('conversion_rate', 'mean')
).reset_index()
display(final_summary)


## 3. 마지막 10분 압축

```text
SQL JOIN 문제:
필요한 컬럼 → 컬럼이 있는 테이블 → 연결키 → JOIN ON → GROUP BY

국가별 매출:
customers.country
orders.customerNumber / orders.orderNumber
orderdetails.quantityOrdered * orderdetails.priceEach

customers → orders → orderdetails
```

```text
pandas 문제:
복사 = df = exam_df.copy()
앞부분 = df.head()
정보 = df.info()
결측치 = df.isna().sum()
그룹 평균 = df.groupby('컬럼')['값'].mean()
교차표 = pd.crosstab(a, b)
날짜 변환 = pd.to_datetime()
```
